# 03 — Logistic Growth Model (S-Curve) Fitting

Fits $P(t) = \frac{K}{1 + e^{-r(t - t_0)}}$ for each country.
Predicts market saturation and death year for traditional PBX.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.models.logistic_growth import (
    fit_all_countries, predict_years, plot_all_fits, summarize_results
)

In [ ]:
panel = pd.read_csv('data/processed/panel_data.csv')
print(f"Panel loaded: {panel.shape[0]} rows, {panel.shape[1]} cols")
print(f"Countries: {panel['country'].unique().tolist()}")

## 3.1 Fit S-Curve for All Countries

In [ ]:
results = fit_all_countries(
    panel,
    penetration_col='fixed_subs_value',
    death_threshold=0.3,  # "dead" = penetration < 30% of peak (≈70% decline); same threshold as notebook 04 so the two reconcile
)
print(f"Fitted {len(results)} countries.")
summary = summarize_results(results)
summary

**EN — Reading the fit table.** Each market is now labelled with a `phase` (growing / declining / flat), an observed `peak_year`/`peak_value`, a `saturation_year` (growth milestone) and a `death_year` (decline milestone). **Important:** for markets that are already declining across the whole window, the *growth* parameters `K`, `r`, `t0` are not interpretable (the growth S-curve is the wrong shape for monotonic decline, so `K` can be huge); rely on `death_year`, `r_squared` and `rmse` instead.

**繁中 — 配適表判讀。** 每個市場現在標註 `phase`（成長／衰退／持平）、觀測 `peak_year`/`peak_value`、`saturation_year`（成長里程碑）與 `death_year`（衰退里程碑）。**重要：**對於在整個觀測窗內已持續衰退的市場，*成長*參數 `K`、`r`、`t0` 不可解讀（成長 S 曲線不適用於單調衰退，故 `K` 可能極大）；應改以 `death_year`、`r_squared`、`rmse` 為準。

## 3.2 Visualize All Fits

In [ ]:
fig = plot_all_fits(results, panel, penetration_col='fixed_subs_value', n_cols=3)
plt.savefig('data/processed/logistic_fits_all.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.3 Results Analysis

In [ ]:
converged = summary[summary['converged']]
print(f"Converged: {len(converged)}/{len(summary)} countries\n")
print("=== Earliest predicted death years ===")
death = converged.dropna(subset=['death_year']).sort_values('death_year')
for _, row in death.head(5).iterrows():
    print(f"  {row['country'].upper()}: {row['death_year']} (R²={row['r_squared']})")
print("\n=== Latest predicted death years ===")
for _, row in death.tail(5).iterrows():
    print(f"  {row['country'].upper()}: {row['death_year']} (R²={row['r_squared']})")
print("\n=== Best fit (highest R²) ===")
best = converged.sort_values('r_squared', ascending=False).head(3)
for _, row in best.iterrows():
    print(f"  {row['country'].upper()}: R²={row['r_squared']}, death={row['death_year']}")

**EN — Death-year ranking.** `death_year` is the year penetration is projected to fall below **30% of each market's historical peak** (≈70% decline = legacy PBX no longer the primary access technology), estimated from the **post-peak decline fit** (not the growth crossing). This is the same threshold the survival model in notebook 04 uses, so the two notebooks agree on which markets sunset first.

**繁中 — 消亡年排名。** `death_year` 為滲透率預估跌破**各市場歷史高峰 30%**（≈70% 衰退＝傳統 PBX 不再是主要接取技術）的年份，係由**峰值後的衰退配適**估計（非成長交叉點）。此門檻與筆記本 04 的存活模型一致，故兩份筆記本對於哪些市場最先退場的結論一致。

## 3.4 Market Phase Classification

In [ ]:
current_year = 2026
def classify_market(row):
    if not row['converged'] or pd.isna(row['death_year']):
        return 'Unknown'
    if row['death_year'] <= current_year:
        return 'Dead/Declining'
    elif row['death_year'] <= current_year + 10:
        return 'Near Death (≤10yr)'
    else:
        return 'Fading (>10yr)'

converged = converged.copy()
converged['market_phase'] = converged.apply(classify_market, axis=1)
phase_counts = converged['market_phase'].value_counts()
print("\nMarket Phase Distribution:")
for phase, count in phase_counts.items():
    print(f"  {phase}: {count}")
print("\nPer-country breakdown:")
print(converged[['country', 'K', 'r', 't0', 'death_year', 'r_squared', 'market_phase']].to_string(index=False))

**EN — Market phase classification.** Buckets each market by how soon `death_year` arrives relative to 2026. 'Dead/Declining' = already past sunset; 'Near Death (≤10yr)' = imminent; 'Fading (>10yr)' = a longer tail of maintenance/replacement revenue.

**繁中 — 市場階段分類。** 依 `death_year` 相對 2026 年的遠近分桶。'Dead/Declining'＝已過退場點；'Near Death (≤10yr)'＝即將退場；'Fading (>10yr)'＝仍有較長的維運/汰換營收尾段。

In [ ]:
# Save results
converged.to_csv('data/processed/logistic_results.csv', index=False)
print("Results saved to data/processed/logistic_results.csv")